# Experiment 1 (error position): human data

Reads the file `npm run getdata` writes (`data/real-all-main-data.json`, gitignored) and turns it
into one tidy row per trial, joined to the stimulus pool.

**The question:** can people tell which order-of-operations misconception a student holds from the
student's work, and does it matter whether the error is at step 1 or step 3?

**Shape of the raw file** (checked 2026-09-18): a list of records, each `{id, data}`. Inside `data`:

| field | what it holds |
|---|---|
| `seedID` | joins to the recruitment file's `session_id` to get `prolific_id` |
| `recruitmentService` | `prolific` for real participants, `web` for a link opened directly |
| `starttimeLocal`, `endtimeLocal` | ISO timestamps |
| `done` | reached the end |
| `pageData_exp.visit_0.data` | 25 entries: the 24 trials, then the bonus block |
| `pageData_practice.visit_0.data` | the 3 practice trials |
| `pageData_quiz`, `pageData_strategy`, `pageData_feedback`, `pageData_demograph` | one entry each |

Each trial entry carries the whole item (`id`, `expression`, `trace`, `error_position`,
`misconceptions`, `probed_misconception`, `category`, `statement_correct`, `student_name`,
`belief_statement`) plus the response (`response` yes/no, `response_method`, `rt` ms,
`responded_agree`, `correct_agree`, `is_correct`, `mouse`).

**Never commit the data file or anything derived from it that contains demographics.**

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO = Path.cwd().parent if Path.cwd().name == "analysis-human" else Path.cwd()
DATA = REPO / "data" / "real-all-main-data.json"
POOL = REPO / "base-task" / "stimulus_pool.json"
RECRUITMENT = REPO / "data" / "private" / "real-main-recruitment.json"  # optional

# ── exclusions ──────────────────────────────────────────────────────────
# Runs by the team before launch. Keep the list; it is the audit trail.
TEST_RUNS = {
    "1ca448ea-1b38-4f0f-960b-9ded32b5539e": "user's own walk-through, 2026-09-18 16:52 EDT",
}
DROP_TEST_RUNS = False  # set True once real participants are in the file
LAUNCH_CUTOFF = None  # e.g. "2026-09-19" to keep only runs starting on or after that date
REQUIRE_DONE = False  # set True to keep only completed runs

SHORT = {
    "add_before_mul": "add<×",
    "add_before_div": "add<÷",
    "sub_before_mul": "sub<×",
    "sub_before_div": "sub<÷",
    "same_priority_rtl": "RTL",
    "outside_bracket_first": "outside()",
}
RULES = list(SHORT)
print(DATA, DATA.exists())

In [ ]:
# ── load and flatten ────────────────────────────────────────────────────
records = json.loads(DATA.read_text())


def entries(rec, page):
    """The recorded entries for one page of one record, or []."""
    return (rec.get(page) or {}).get("visit_0", {}).get("data", []) or []


people, trials = [], []
for raw in records:
    d = raw["data"]
    rows = [r for r in entries(d, "pageData_exp") if "id" in r]
    bonus = next((r for r in entries(d, "pageData_exp") if r.get("phase")), {})
    people.append(
        {
            "seedID": d.get("seedID"),
            "service": d.get("recruitmentService"),
            "done": bool(d.get("done")),
            "start": pd.to_datetime(d.get("starttimeLocal"), errors="coerce", utc=True),
            "end": pd.to_datetime(d.get("endtimeLocal"), errors="coerce", utc=True),
            "n_trials": len(rows),
            "n_practice": len(entries(d, "pageData_practice")),
            "client_accuracy": bonus.get("accuracy"),
            "client_bonus": bonus.get("bonus"),
            "strategy": (entries(d, "pageData_strategy") or [{}])[0],
        }
    )
    for i, r in enumerate(rows):
        trials.append(
            {
                "seedID": d.get("seedID"),
                "trial": i + 1,
                "item": r["id"],
                "category": r["category"],
                "position": r["error_position"],
                "misconception": r["misconceptions"][0],
                "named": r["probed_misconception"],
                "statement_correct": bool(r["statement_correct"]),
                "response": r.get("response"),
                "method": r.get("response_method"),
                "rt": r.get("rt"),
                "n_mouse": len(r.get("mouse") or []),
                "expression": r["expression"],
            }
        )

people = pd.DataFrame(people)
trials = pd.DataFrame(trials)

# score from the raw response, never trusting the client's is_correct
trials["said_yes"] = trials["response"].eq("yes")
trials["correct"] = trials["said_yes"].eq(trials["statement_correct"])
trials["rule"] = trials["misconception"].map(SHORT)
trials["named_rule"] = trials["named"].map(SHORT)

print(f"{len(people)} records, {len(trials)} trials")
people[["seedID", "service", "done", "start", "n_trials", "client_accuracy", "client_bonus"]]

In [ ]:
# ── who counts ──────────────────────────────────────────────────────────
keep = people.copy()
keep["is_test"] = keep["seedID"].isin(TEST_RUNS)
if DROP_TEST_RUNS:
    keep = keep[~keep["is_test"]]
if LAUNCH_CUTOFF:
    keep = keep[keep["start"] >= pd.Timestamp(LAUNCH_CUTOFF, tz="UTC")]
if REQUIRE_DONE:
    keep = keep[keep["done"]]
keep = keep[keep["n_trials"] > 0]

for sid, why in TEST_RUNS.items():
    mark = "dropped" if DROP_TEST_RUNS else "KEPT (DROP_TEST_RUNS is False)"
    print(f"test run {sid[:8]}… {mark}: {why}")

df = trials[trials["seedID"].isin(keep["seedID"])].copy()
print(f"\nanalysing {keep.seedID.nunique()} participants, {len(df)} trials")

In [ ]:
# ── join the pool, so foil_status and the observer's marginal are available ──
pool = pd.DataFrame(json.loads(POOL.read_text()))[
    ["id", "foil_status", "io_foil_marginal", "n_ops"]
].rename(columns={"id": "item"})
df = df.merge(pool, on="item", how="left", validate="many_to_one")

missing = df["n_ops"].isna().sum()
print(f"trials whose item is not in the current pool: {missing}")
if missing:
    print("  (the pool was rebuilt after these were run, so those trials cannot be joined)")
    print(sorted(df.loc[df['n_ops'].isna(), 'item'].unique())[:10])

In [ ]:
# ── integrity: every participant should have the same balanced form ─────
def form_check(g):
    return pd.Series(
        {
            "trials": len(g),
            "distinct items": g["item"].nunique(),
            "YES trials": int(g["statement_correct"].sum()),
            "step 1": int((g["position"] == 1).sum()),
            "per misconception": sorted(g["misconception"].value_counts().unique().tolist()),
            "answered by key": int((g["method"] == "key").sum()),
            "unanswered": int(g["response"].isna().sum()),
        }
    )


checks = df.groupby("seedID").apply(form_check, include_groups=False)
print("want: 24 trials, 24 distinct items, 12 YES, 12 step-1, [4] per misconception, 24 by key, 0 unanswered")
checks

In [ ]:
# ── headline: accuracy by error position (the manipulated factor) ───────
def acc(g):
    return pd.Series({"n": len(g), "correct": int(g["correct"].sum()), "accuracy": g["correct"].mean()})


print("overall:", f"{df['correct'].mean():.1%}", f"({int(df['correct'].sum())}/{len(df)})\n")
print("by error position")
print(df.groupby("position").apply(acc, include_groups=False), "\n")
print("by position x whether the statement was true")
print(df.groupby(["position", "statement_correct"]).apply(acc, include_groups=False))

In [ ]:
# ── per participant, so the position effect can be tested within subject ──
per_person = (
    df.pivot_table(index="seedID", columns="position", values="correct", aggfunc="mean")
    .rename(columns={1: "step 1", 3: "step 3"})
)
per_person["difference"] = per_person["step 1"] - per_person["step 3"]
display(per_person)

if len(per_person) >= 8:
    from scipy import stats

    t, p = stats.ttest_rel(per_person["step 1"], per_person["step 3"])
    print(f"paired t-test on the position effect: t = {t:.2f}, p = {p:.3f}")
else:
    print("too few participants for a test yet; this is the table to test later")

In [ ]:
# ── accuracy by misconception, and by the statement that was named ──────
by_rule = df.groupby("rule").apply(acc, include_groups=False).reindex([SHORT[r] for r in RULES])
print("misconception actually in the work")
print(by_rule, "\n")
print("misconception the statement named")
print(df.groupby("named_rule").apply(acc, include_groups=False).reindex([SHORT[r] for r in RULES]), "\n")
print("disagree trials by how the work treats the named rule (recorded, not balanced)")
print(df[df["category"] == "B"].groupby("foil_status").apply(acc, include_groups=False))

In [ ]:
# ── yes-bias: hits, false alarms, d' ────────────────────────────────────
from scipy.stats import norm


def sdt(g):
    yes_trials, no_trials = g[g["statement_correct"]], g[~g["statement_correct"]]
    # log-linear correction, so 0 and 1 rates stay finite
    hit = (yes_trials["said_yes"].sum() + 0.5) / (len(yes_trials) + 1)
    fa = (no_trials["said_yes"].sum() + 0.5) / (len(no_trials) + 1)
    return pd.Series(
        {
            "hit rate": hit,
            "false alarm": fa,
            "d'": norm.ppf(hit) - norm.ppf(fa),
            "criterion": -0.5 * (norm.ppf(hit) + norm.ppf(fa)),
            "said yes": g["said_yes"].mean(),
        }
    )


print("overall")
print(sdt(df).round(3), "\n")
print("by error position")
print(df.groupby("position").apply(sdt, include_groups=False).round(3))

In [ ]:
# ── time on task ────────────────────────────────────────────────────────
rt = df[df["rt"].notna()].copy()
rt["rt_s"] = rt["rt"] / 1000
print("reaction time in seconds (the answer keys unlock after 3 s, so 3 s is the floor)")
print(rt["rt_s"].describe()[["count", "min", "50%", "mean", "max"]].round(2), "\n")
print("median rt by position")
print(rt.groupby("position")["rt_s"].median().round(2), "\n")
print("median rt by correctness")
print(rt.groupby("correct")["rt_s"].median().round(2))

if "end" in keep and keep["end"].notna().any():
    dur = (keep["end"] - keep["start"]).dt.total_seconds() / 60
    print("\nminutes from start to finish:", dur.round(1).tolist())

In [ ]:
# ── figures ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

g = df.groupby(["position", "statement_correct"])["correct"].mean().unstack()
g.index = ["step 1", "step 3"]
g.columns = ["statement false (NO)", "statement true (YES)"]
g.plot.bar(ax=axes[0], rot=0, color=["#eb6834", "#2a78d6"])
axes[0].set_title("accuracy by error position")
axes[0].set_ylabel("proportion correct")
axes[0].axhline(0.5, ls="--", lw=0.9, color="#999")
axes[0].set_ylim(0, 1)

by_rule["accuracy"].plot.bar(ax=axes[1], rot=30, color="#2a78d6")
axes[1].set_title("accuracy by misconception in the work")
axes[1].axhline(0.5, ls="--", lw=0.9, color="#999")
axes[1].set_ylim(0, 1)

axes[2].hist(rt["rt_s"], bins=20, color="#2a78d6")
axes[2].set_title("reaction time (s)")
axes[2].axvline(3, ls="--", lw=0.9, color="#999")

for ax in axes[:2]:
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
# ── bonuses, recomputed from the raw responses ──────────────────────────
MAX_BONUS, CHANCE = 2.0, 0.5
bonus = df.groupby("seedID")["correct"].agg(["size", "sum", "mean"]).rename(
    columns={"size": "n", "sum": "correct", "mean": "accuracy"}
)
bonus["bonus"] = (
    ((bonus["accuracy"] - CHANCE) / (1 - CHANCE)).clip(lower=0) * MAX_BONUS
).round(2)
bonus = bonus.join(people.set_index("seedID")[["client_bonus"]])
bonus["matches client"] = np.isclose(bonus["bonus"], bonus["client_bonus"].astype(float))
display(bonus)

if RECRUITMENT.exists():
    rec = json.loads(RECRUITMENT.read_text())
    ids = {r["session_id"]: r.get("prolific_id") for r in (rec if isinstance(rec, list) else rec.values())}
    bonus["prolific_id"] = bonus.index.map(ids)
    print("\npaste into Prolific's bulk bonus box (prolific_id,amount):")
    for sid, row in bonus.iterrows():
        if row.get("prolific_id") and row["bonus"] > 0:
            print(f"{row['prolific_id']},{row['bonus']:.2f}")
else:
    print(f"\nno recruitment file at {RECRUITMENT}; run `npm run getrecruitment` to map seedID -> prolific_id")
print("\nKeep data/private/bonus_paid.csv as the ledger and never pay twice from this table alone.")

In [ ]:
# ── tidy export for modelling elsewhere ─────────────────────────────────
# data/private/ is gitignored; participant responses must never be committed.
out = REPO / "data" / "private" / "exp1_trials.csv"
out.parent.mkdir(parents=True, exist_ok=True)
df.drop(columns=["expression"]).to_csv(out, index=False)
print("wrote", out, f"({len(df)} rows)")
df.head()